In [5]:
import sys
sys.argv.extend(["--max-memory", "8G"])

import datetime as dt
from pathlib import Path

import geopandas as gpd
import pandas as pd
import r5py
from shapely.geometry import Point

TRANSPORTE = Path("../data/raw/transporte")
LIMPIOS = TRANSPORTE / "limpios"
PROCESSED = Path("../data/processed")

network = r5py.TransportNetwork(
    TRANSPORTE / "madrid-261127.osm.pbf",
    [
        LIMPIOS / "gtfs_metro_2026.zip",
        LIMPIOS / "gtfs_metro_ligero.zip",
        LIMPIOS / "gtfs_cercanias.zip",
        LIMPIOS / "gtfs_emt.zip",
    ],
)
print("✅ Red cargada")

✅ Red cargada


In [6]:
barrios = gpd.read_parquet(PROCESSED / "barrios.parquet")

origenes = gpd.GeoDataFrame(
    {"id": barrios["cod_barrio"].values},
    geometry=barrios.representative_point().values,
    crs=barrios.crs,
).to_crs(4326)

ZONAS_TRABAJO = {
    "Nuevos Ministerios / AZCA": (40.4466, -3.6920),
    "Plaza de Castilla / Cuatro Torres": (40.4770, -3.6880),
    "Sol / Centro": (40.4169, -3.7035),
    "Gran Vía / Callao": (40.4200, -3.7058),
    "Atocha": (40.4065, -3.6895),
    "Méndez Álvaro": (40.3955, -3.6780),
    "Moncloa": (40.4350, -3.7190),
    "Ciudad Universitaria": (40.4450, -3.7280),
    "Las Tablas": (40.5050, -3.6700),
    "IFEMA / Campo de las Naciones": (40.4630, -3.6160),
    "Julián Camarillo (San Blas)": (40.4400, -3.6250),
    "Aeropuerto T4": (40.4910, -3.5930),
    "Hospital La Paz": (40.4810, -3.6870),
    "Hospital 12 de Octubre": (40.3750, -3.6990),
}
destinos = gpd.GeoDataFrame(
    {"id": list(ZONAS_TRABAJO)},
    geometry=[Point(lon, lat) for lat, lon in ZONAS_TRABAJO.values()],
    crs=4326,
)
print(len(origenes), "barrios ×", len(destinos), "zonas =", len(origenes) * len(destinos), "trayectos")

131 barrios × 14 zonas = 1834 trayectos


In [7]:
import time

rejilla = gpd.read_parquet(PROCESSED / "rejilla_destinos.parquet")[["id", "geometry"]]
print(f"{len(origenes)} barrios × {len(rejilla)} puntos")

modos = [
    (r5py.TransportMode.BICYCLE, "min_bici"),
    (r5py.TransportMode.CAR, "min_coche"),
    (r5py.TransportMode.WALK, "min_andando"),
]

resultados = []
for modo, columna in modos:
    t0 = time.time()
    ttm = r5py.TravelTimeMatrix(
        network,
        origins=origenes,
        destinations=rejilla,
        departure=dt.datetime(2026, 12, 1, 8, 30),
        transport_modes=[modo],
        max_time=dt.timedelta(minutes=120),
    )
    df = (
        pd.DataFrame(ttm)
        .rename(columns={"from_id": "cod_barrio", "to_id": "id_destino", "travel_time": columna})
        .set_index(["cod_barrio", "id_destino"])
    )
    resultados.append(df)
    print(f"{columna}: {time.time() - t0:.0f} s · sin tiempo: {df[columna].isna().mean():.1%}")

otros_modos = pd.concat(resultados, axis=1).reset_index()
otros_modos.to_parquet(PROCESSED / "tiempos_otros_modos.parquet")
print("✅ Guardado")
otros_modos.describe()

131 barrios × 1333 puntos
min_bici: 133 s · sin tiempo: 11.3%
min_coche: 229 s · sin tiempo: 2.3%
min_andando: 41 s · sin tiempo: 73.5%
✅ Guardado


,id_destino,min_bici,min_coche,min_andando
count,174623.000000,154853.000000,170654.000000,46296.000000
mean,666.000000,66.910186,19.788455,79.889666
std,384.804948,27.682384,8.847120,28.337747
min,0.000000,0.000000,0.000000,1.000000
25%,333.000000,46.000000,15.000000,60.000000
50%,666.000000,67.000000,19.000000,84.000000
75%,999.000000,88.000000,23.000000,104.000000
max,1332.000000,140.000000,126.000000,142.000000
